# IBM Backend Benchmarking: Device Comparison

This notebook demonstrates a simple workflow for benchmarking Sudoku quantum circuits on IBM Quantum backends. We'll compare two devices using environment variables for secure credential management.

## Prerequisites

1. **IBM Quantum Account**: Sign up at [https://quantum.ibm.com/](https://quantum.ibm.com/)
2. **Dependencies**: Ensure `qiskit-ibm-runtime` is installed
3. **Environment Variables**: Set the following:
   - `IBM_TOKEN`: Your IBM Quantum API token
   - `IBM_INSTANCE`: (Optional) Your IBM Quantum instance/CRN

## Workflow Overview

1. Import libraries and configure authentication
2. Create a simple 2×2 Sudoku puzzle
3. Set up quantum solver with simple encoding
4. Compare execution on two IBM backends
5. Analyze and visualize results

## 1. Import Required Libraries

In [2]:
import os
from sudoku_nisq import QSudoku, ExactCoverQuantumSolver
from sudoku_nisq.backends import BackendManager

# Visualization and data handling
import matplotlib.pyplot as plt
import pandas as pd

## 2. Configure IBM Authentication

Load credentials from environment variables. Set these before running:
```bash
export IBM_TOKEN="your_token_here"
export IBM_INSTANCE="your_instance_here"
```

In [1]:
# Load credentials from environment
ibm_token = "sMwvHM0JfNSKuM17EeIT07gxmLzn7J4D-w0JF2GKgyEM"
ibm_instance = "crn:v1:bluemix:public:quantum-computing:us-east:a/53bccd1b6f1943a486285adb9d2dfa3f:ac903e4d-622b-41dd-8747-b33565ec1be8::"

## 3. Authenticate and List Available Devices

In [ ]:
# Get BackendManager singleton and authenticate
manager = BackendManager.inst()

try:
    devices = manager.authenticate_ibm(
        api_token=ibm_token,
        instance=ibm_instance
    )
    print(f"✓ Authenticated with IBM Quantum")
    print(f"  Available devices: {', '.join(devices[:5])}")
    if len(devices) > 5:
        print(f"  ... and {len(devices) - 5} more")
except Exception as e:
    print(f"✗ Authentication failed: {e}")
    raise

## 4. Select Devices for Comparison

Choose two IBM backends to compare. Update these device names based on your available devices.

In [ ]:
# Select two devices to compare (update these based on available devices)
device1 = os.getenv("IBM_DEVICE_1", "ibm_brisbane")
device2 = os.getenv("IBM_DEVICE_2", "ibm_kyoto")

# Register backends with aliases
try:
    manager.add_backend("ibm", device1, alias="device1")
    manager.add_backend("ibm", device2, alias="device2")
    print(f"✓ Registered backends:")
    print(f"  Device 1: {device1}")
    print(f"  Device 2: {device2}")
except Exception as e:
    print(f"✗ Failed to register backends: {e}")
    print(f"  Available devices: {devices}")
    raise

## 5. Create Sudoku Puzzle

Create a simple 2×2 Sudoku puzzle for quick testing on real hardware.

In [ ]:
# Generate a 2×2 Sudoku puzzle
puzzle = QSudoku.generate(size=4, num_missing_cells=2, canonicalize=True)

print("✓ Puzzle created")
print(f"  Size: 2×2 (4 cells)")
print(f"  Missing cells: 2")
print(f"  Puzzle hash: {puzzle._puzzle.get_hash()[:8]}...")
print(f"\nPuzzle:")
print(puzzle._puzzle)

## 6. Configure Quantum Solver

Set up the exact cover quantum solver with simple encoding.

In [ ]:
# Configure solver with simple encoding
puzzle.set_solver(ExactCoverQuantumSolver, encoding="simple")

# Get resource estimation
resources = puzzle._solver.resource_estimation()
print("✓ Solver configured")
print(f"  Encoding: simple")
print(f"  Estimated resources:")
print(f"    Qubits: {resources['n_qubits']}")
print(f"    Gates: {resources['n_gates']}")
print(f"    Depth: {resources.get('depth', 'N/A')}")

## 7. Build and Inspect Circuit

Build the quantum circuit using Qiskit (required for IBM backends).

In [ ]:
# Build circuit with Qiskit SDK
circuit = puzzle.build_circuit(sdk="qiskit")

print("✓ Circuit built")
print(f"  SDK: Qiskit")
print(f"  Qubits: {circuit.num_qubits}")
print(f"  Classical bits: {circuit.num_clbits}")
print(f"\nCircuit summary:")
print(circuit)

## 8. Run on Device 1

Execute the circuit on the first IBM backend. This will submit a job to real quantum hardware.

In [ ]:
# Run on first device
shots = 1024
print(f"Running on {device1} with {shots} shots...")
print("(This may take several minutes depending on queue)")

result1 = puzzle.run("device1", shots=shots, opt_level=1)

print(f"\n✓ Execution complete on {device1}")
print(f"  Total counts: {sum(result1.values())}")

## 9. Run on Device 2

Execute the same circuit on the second IBM backend for comparison.

In [ ]:
# Run on second device
print(f"Running on {device2} with {shots} shots...")
print("(This may take several minutes depending on queue)")

result2 = puzzle.run("device2", shots=shots, opt_level=1)

print(f"\n✓ Execution complete on {device2}")
print(f"  Total counts: {sum(result2.values())}")

## 10. Format and Compare Results

Analyze results from both devices using the built-in metrics formatting.

In [ ]:
# Format results for both devices
formatted1 = puzzle.format_result(result1)
formatted2 = puzzle.format_result(result2)

# Create comparison dataframe
comparison_data = {
    "Metric": [
        "Success Rate",
        "Distinct Solutions",
        "Top-1 Mass",
        "Top-3 Mass",
        "Efficiency (η)",
        "Efficiency (η²)",
        "Total Gates",
        "2-Qubit Gates",
        "Circuit Depth",
        "Shots for 99%",
    ],
    device1: [
        f"{formatted1['success_rate']:.2%}",
        formatted1['distinct_solutions_observed'],
        f"{formatted1['topk_solution_mass'][0]:.2%}",
        f"{formatted1['topk_solution_mass'][2]:.2%}",
        f"{formatted1['eta']:.4f}",
        f"{formatted1['eta2']:.4f}",
        formatted1['G_total'],
        formatted1['G_2q'],
        formatted1['depth'],
        formatted1['shots_for_99pct'],
    ],
    device2: [
        f"{formatted2['success_rate']:.2%}",
        formatted2['distinct_solutions_observed'],
        f"{formatted2['topk_solution_mass'][0]:.2%}",
        f"{formatted2['topk_solution_mass'][2]:.2%}",
        f"{formatted2['eta']:.4f}",
        f"{formatted2['eta2']:.4f}",
        formatted2['G_total'],
        formatted2['G_2q'],
        formatted2['depth'],
        formatted2['shots_for_99pct'],
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("DEVICE COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

## 11. Visualize Results - Device 1

In [ ]:
# Visualize first device results
puzzle.counts_plot(result1, backend_alias=device1)

## 12. Visualize Results - Device 2

In [ ]:
# Visualize second device results
puzzle.counts_plot(result2, backend_alias=device2)

## 13. Side-by-Side Metric Comparison

Create a visual comparison of key performance metrics.

In [ ]:
# Create bar chart comparison for key metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Success Rate
ax = axes[0, 0]
ax.bar([device1, device2], 
       [formatted1['success_rate'], formatted2['success_rate']],
       color=['#1f77b4', '#ff7f0e'])
ax.set_ylabel('Success Rate')
ax.set_title('Success Rate Comparison')
ax.set_ylim([0, 1])
for i, v in enumerate([formatted1['success_rate'], formatted2['success_rate']]):
    ax.text(i, v + 0.02, f"{v:.2%}", ha='center', va='bottom')

# Efficiency (η)
ax = axes[0, 1]
ax.bar([device1, device2],
       [formatted1['eta'], formatted2['eta']],
       color=['#1f77b4', '#ff7f0e'])
ax.set_ylabel('Efficiency (η)')
ax.set_title('Efficiency η Comparison')
for i, v in enumerate([formatted1['eta'], formatted2['eta']]):
    ax.text(i, v + 0.01, f"{v:.4f}", ha='center', va='bottom')

# Distinct Solutions
ax = axes[1, 0]
ax.bar([device1, device2],
       [formatted1['distinct_solutions_observed'], formatted2['distinct_solutions_observed']],
       color=['#1f77b4', '#ff7f0e'])
ax.set_ylabel('Count')
ax.set_title('Distinct Solutions Observed')
for i, v in enumerate([formatted1['distinct_solutions_observed'], formatted2['distinct_solutions_observed']]):
    ax.text(i, v + 0.1, str(v), ha='center', va='bottom')

# 2-Qubit Gates
ax = axes[1, 1]
ax.bar([device1, device2],
       [formatted1['G_2q'], formatted2['G_2q']],
       color=['#1f77b4', '#ff7f0e'])
ax.set_ylabel('Count')
ax.set_title('2-Qubit Gates (Transpiled)')
for i, v in enumerate([formatted1['G_2q'], formatted2['G_2q']]):
    ax.text(i, v + 1, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Summary and Next Steps

This notebook demonstrated:

1. **Authentication**: Using environment variables for secure IBM Quantum access
2. **Device Selection**: Comparing two IBM backends side-by-side
3. **Circuit Execution**: Running the same Sudoku circuit on real quantum hardware
4. **Metrics Analysis**: Evaluating success rate, efficiency, and resource usage
5. **Visualization**: Creating plots and comparisons for benchmark results

### Key Metrics Explained

- **Success Rate** (p_succ): Probability of measuring a valid solution
- **Efficiency η**: Ratio of valid solution probability to uniform distribution
- **Efficiency η²**: Squared efficiency for multi-solution problems
- **Distinct Solutions**: Number of different valid solutions observed
- **G_2q, Depth**: Transpiled circuit complexity for the specific backend

### Next Steps

- Try different puzzle sizes (4×4 for more challenging benchmarks)
- Compare `simple` vs. `pattern` encoding
- Run with higher shot counts for better statistics
- Test with different optimization levels (0-3)
- Use Aer simulation with noise models to predict hardware behavior
- Explore error mitigation techniques (ZNE, PEC)

For more information, see the [documentation](../docs/guide/) and [metrics reference](../docs/guide/metrics_reference.md).